# 2.Runnable回退机制

知识点讲解：with_fallbacks() 回退机制提升系统可用性

1. 组件定位：什么是回退（Fallback）机制
   - 在某些场合中，对于 Runnable 组件的出错，并不想执行重试方案，
     而是执行特定的备份/回退方案（例如 OpenAI 的 LLM 异常时，自动切换到文心一言）
   - LangChain 提供对应的回退机制——with_fallback()：当主组件失败时，自动切换到备用组件
   - 返回 RunnableWithFallbacks，仍实现 Runnable 协议，可参与 | 管道组合
   - 属于容错设计中的降级策略，目标是保证服务"可用"而非"最优"
   - 与重试的区别：重试是"再试同一个"，回退是"换一个试"

2. with_fallbacks() 的核心参数
   - fallbacks: Sequence[Runnable] - 备用组件列表，按顺序依次尝试，必填
   - exceptions_to_handle: tuple - 触发回退的异常类型，默认 (Exception,)
   - exception_key: str - 可选，将异常信息以该键传入备用组件的输入

3. 回退的执行顺序与失败传播
   - 先执行主组件；成功则直接返回，不触碰备用组件
   - 主组件失败 → 尝试 fallbacks[0] → 仍失败 → 尝试 fallbacks[1] → ...
   - 全部失败时，抛出最后一个组件的异常
   - 可注册多级回退，形成"主 → 备1 → 备2"的降级链条

4. 本示例的容错场景
   - 主模型故意写为不存在的 deepseek-v4-pro-18k，调用必然失败
   - 备用模型为百度文心一言 QianfanChatEndpoint
   - 实际效果：主模型报错后自动切换到文心一言，用户侧仍能得到正常回复

5. 底层实现原理
   - 运行流程简单直接：按照主组件 → fallbacks 列表的顺序依次尝试
   - 每个组件执行时捕获 exceptions_to_handle 中指定的异常类型
   - 一旦某个组件成功执行，立即返回结果，不再尝试后续组件
   - 所有组件都失败时，抛出最后一个组件的异常
   - 核心实现在 RunnableWithFallbacks 类的 invoke 方法中

6. 典型的回退设计模式
   - 跨厂商回退：OpenAI 故障时切换到文心一言/通义千问，规避单点风险
   - 降级回退：强模型不可用时降级到轻量模型，保证基本可用性
   - 兜底回退：所有模型都失败时返回预设的固定文案，避免直接报错
   - 配额回退：主账号限流时切换到备用 API Key

7. 与 configurable_alternatives 的对比
   - with_fallbacks：被动触发，仅在异常发生时自动切换，调用方无感
   - configurable_alternatives：主动选择，由调用方通过 config 显式指定
   - 两者正交，可叠加使用，分别解决"容错"与"路由"两类问题

8. 注意事项
   - 备用组件必须与主组件的输入输出类型兼容，否则回退后链会断裂
   - 建议用 exceptions_to_handle 精确限定异常类型，避免掩盖代码 bug
   - 回退会叠加延迟（主组件失败耗时 + 备用组件耗时），需评估超时预算
   - 回退发生时应记录日志与监控告警，否则故障会被静默吞掉

9. 数据流转过程
   - {"query": "你好，你是?"} → prompt → Messages
   - → RunnableWithFallbacks 主组件 ChatOpenAI("deepseek-v4-pro-18k") → 抛出异常
   - → 捕获异常，切换至 QianfanChatEndpoint() → AIMessage
   - → StrOutputParser → str


In [ ]:
import dotenv
from langchain_community.chat_models.baidu_qianfan_endpoint import QianfanChatEndpoint
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
prompt = ChatPromptTemplate.from_template("{query}")
llm = ChatOpenAI(model="deepseek-v4-pro-18k").with_fallbacks([QianfanChatEndpoint()])
chain = prompt | llm | StrOutputParser()
content = chain.invoke({"query": "你好，你是?"})
print(content)